# Learning STL Decomposition — Gig-Worker Income Mini Project

**STL (Seasonal-Trend decomposition using LOESS)** splits a time series into three parts:

`income = trend + seasonal + residual`

The idea: a gig worker's income swings a lot week to week, but a chunk of that swing is
*predictable* (festival season, monsoon dips, weekly demand cycles). STL lets us separate
that predictable part from the leftover, unexplained noise — which is arguably the part
that actually matters for assessing risk.

This notebook simulates one gig worker's weekly income over 2 years and decomposes it.

In [ ]:
!pip install statsmodels --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

plt.rcParams['figure.dpi'] = 110

## Step 1 — Simulate a gig worker's weekly income

Built from four pieces: a base income level, a slow trend, a seasonal pattern (festival bump + monsoon dip), and random noise + occasional income shocks.

In [ ]:
def simulate_income(n_weeks=104, base_income=6000, seasonal_amplitude=1400,
                     noise_std=350, shock_prob=0.03, seed=7):
    rng = np.random.default_rng(seed)
    weeks = np.arange(n_weeks)

    # seasonal: annual cycle with festival bump (~week 43) and monsoon dip (~week 26)
    annual_cycle = np.sin(2 * np.pi * (weeks - 13) / 52)
    festival_bump = 0.6 * np.exp(-0.5 * ((weeks % 52 - 43) / 4) ** 2)
    monsoon_dip = -0.5 * np.exp(-0.5 * ((weeks % 52 - 26) / 5) ** 2)
    seasonal = seasonal_amplitude * (annual_cycle * 0.4 + festival_bump + monsoon_dip)

    noise = rng.normal(0, noise_std, n_weeks)

    shocks = np.zeros(n_weeks)
    shock_weeks = rng.random(n_weeks) < shock_prob
    shocks[shock_weeks] = -rng.normal(2500, 1000, shock_weeks.sum()).clip(0)

    income = base_income + seasonal + noise + shocks
    return pd.Series(np.clip(income, 0, None), name="weekly_income")

income = simulate_income()
income.plot(figsize=(11, 4), title="Simulated Weekly Income — Gig Worker", color="#0f3460")
plt.ylabel("Income (INR)")
plt.xlabel("Week")
plt.tight_layout()
plt.show()

## Step 2 — Run STL decomposition

`period=52` because we're looking for an annual seasonal cycle in weekly data.

In [ ]:
stl = STL(income, period=52, robust=True)
result = stl.fit()

fig = result.plot()
fig.set_size_inches(11, 7)
fig.suptitle("STL Decomposition of Weekly Gig Income", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## Step 3 — What this actually tells us

- **Trend** — the slow underlying direction of the worker's income.
- **Seasonal** — the repeating, *predictable* part (festival spikes, monsoon dips).
- **Residual** — what's left after removing trend + seasonality. This is the genuinely
  unpredictable part of the income — arguably a better signal of real financial risk than
  raw income variance, which lumps predictable swings and true risk together.

In [ ]:
raw_std = income.std()
residual_std = result.resid.std()

print(f"Raw income std dev:      {raw_std:,.0f}")
print(f"Residual (true-risk) std dev: {residual_std:,.0f}")
print(f"\n{(1 - residual_std/raw_std)*100:.0f}% of the raw volatility turns out to be predictable/seasonal, not true risk.")

---
*Mini project — part of ongoing research on gig-economy income volatility and credit access.*